# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?

**Відповідь:**
Якби ми передали один склеєний рядок замість списку окремих віршів у тренер BPE, алгоритм міг би почати знаходити закономірності та створювати токени на стику двох різних віршів. Це призведе до того, що токенізатору довелося б тримати весь корпус як один гігантський текстовий блок під час підрахунку частот пар, що легко може призвести до помилки нестачі пам'яті.

2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))

**Відповідь: **
Normalization: базове очищення сирого тексту. На цьому етапі видаляються зайві пробіли, текст приводиться до нижнього регістру, а також виконується стандартизація символів.

Pre-tokenization: текст розбивається на базові одиниці (зазвичай слова) за пробілами або знаками пунктуації. Це потрібно для того, щоб алгоритм токенізації не намагався об'єднати частини різних слів у один токен.

Model: основний етап, на якому до отриманих слів застосовується обраний алгоритм (BPE, WordPiece або Unigram). Слова розбиваються на відомі підслова (субтокени) згідно зі словником і перетворюються на числові ідентифікатори (ID).

Post-processing:  формування фінальної послідовності. Тут додаються спеціальні токени (позначання початку чи кінця послідовності, [CLS], [SEP] або <eos>), які вимагає конкретна мовна модель.

2. Що таке `"<unk>", "<eos>"` в 2.1.?

**Відповідь: **
<unk> — це спеціальний токен, який замінює будь-які слова чи символи, що не увійшли до сформованого словника під час тренування токенізатора.
<eos> — це спеціальний токен, який позначає межу між окремими віршами.

3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).


4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?

**Відповідь: **
Токен <eos> позначає межу між окремими віршами. Якщо цей роздільник прибрати, під час конкатенації текстів останній токен одного вірша впритул стикнеться з першим токеном наступного. Через це під час тренування модель помилково сприйматиме різні твори як один суцільний текст і почне заучувати безглузді переходи між ними.

5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?

**Відповідь: **
train_data та val_data — це одновимірні PyTorch-тензори, що містять цілі числа формату int64. Кожен елемент цих тензорів представляє числовий ідентифікатор (ID) окремого токена зі словника, отриманого після кодування тексту. Весь текстовий корпус, перетворений на єдину безперервну послідовність чисел, де окремі вірші розділені ідентифікатором <eos>.

6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.

**Відповідь: **

7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?

**Відповідь: **
Нижньотрикутна матриця забезпечує те, що токен на певній позиції може звертати увагу лише на себе та всі попередні токени. Якби матриця була верхньотрикутною, токени дивилися б лише на майбутні слова і не бачили попереднього контексту. Якби маски не було взагалі, під час тренування модель могла б вільно підглядати на наступні токени.

8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.

**Відповідь: **
Чекпоінт із найкращим val_loss та останній чекпоінт збігаються, якщо помилка на валідаційній вибірці стабільно зменшується аж до останньої ітерації тренування. Вони не збігаються, якщо під час тренування починається перенавчання (overfitting): тоді тренувальна помилка продовжує падати, але валідаційна помилка починає зростати, і модель зберігає свій найкращий стан на одній із попередніх ітерацій.

На моєму графіку крива валідаційної помилки стабільно падає до останньої ітерації, ознак перенавчання немає. Тому в моєму випадку чекпоінт із найкращим val_loss збігається з останнім.


10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?

**Відповідь: **
Зменшення EVAL_INTERVAL з 250 до 50 робить криву навчання детальнішою, оскільки ми отримуємо більше точок вимірювання на графіку. Загальний час тренування збільшився, оскільки функція estimate_loss() тепер викликається в 5 разів частіше, витрачаючи ресурси на прогін EVAL_ITERS батчів. Робити EVAL_INTERVAL ще меншим немає сенсу

до зміни - Тренування завершено за 6.8 хв
Найкращий чекпоінт: iter 7999  (val_loss=5.675)
після зміни - Тренування завершено за 7.2 хв
Найкращий чекпоінт: iter 7999  (val_loss=5.359)

11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.

**Відповідь: **
Параметр temperature ($T$) математично ділить сирі передбачення моделі (логіти, $z_i$) перед тим, як вони пройдуть через функцію softmax. Формула розрахунку ймовірності набуває такого вигляду: $p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$.   При низькій температурі ($T < 1$, наприклад 0.2) різниця між логітами штучно збільшується (оскільки ділення на десятковий дріб еквівалентне множенню). Токен із найбільшим логітом отримує ймовірність, близьку до 100%, тому генерація стає дуже впевненою, консервативною та передбачуваною.При дуже високій температурі ($T > 1$, наприклад 1.4) логіти діляться на число, більше за одиницю, що стискає їх ближче до нуля та одне до одного. Різниця між лідерами та аутсайдерами нівелюється, і після softmax розподіл імовірностей наближається до рівномірного. Модель починає обирати будь-які токени зі словника з майже однаковим шансом, тому текст стає хаотичним і майже випадковим.

12. За що саме відповідає параметр `top_p`?

**Відповідь: **
відповідає за динамічне обмеження пулу токенів, з яких модель робить наступний вибір. Замість того, щоб розглядати весь словник, алгоритм сортує токени за ймовірністю та відбирає лише найімовірніші з них, поки їхня сумарна ймовірність не досягне заданого значення p

13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

**Відповідь: **

до повторного запуску -
--- temperature=0.2, top_p=0.1 ---
Якось уночі студент Житомирської політехніки прокинувся від дивного шуму. Він підвівся з ліжка, зігнувся, намагався зрозуміти, чому так шумить, але здавалося, що нічого не відчуває, крім відчуття тривоги і незвичайної тишини, яка навколо нього виглядала навіть глибшою, ніж звичайно.

--- temperature=0.8, top_p=0.6 ---
Якось уночі студент Житомирської політехніки прокинувся від дивного шуму. Він підвівся з ліжка, зіштовхнувся обертаючийся ковер, відчинив вікно й побачив на підлозі чорну, високу тінь, яка звивалася, як жилистий вінок, і звучала, як якісь незвичайні, навіть незвичайні слова.

--- temperature=1.4, top_p=0.8 ---
Якось уночі студент Житомирської політехніки прокинувся від дивного шуму. Він підвівся з ліжка, витягнув телефон і пошукав, де походить цей стрявін. Ніч настала, вікна були закриті, але шум залишався — мовчив, ніби чекав, коли він з'явиться.

повторний запуск - --- temperature=0.2, top_p=0.1 ---
<think>

У результаті виконання перед самим продовженням речення з'явивмя додатковий блок тексту <think>...</think>, де модель покроково "розмірковує" над сюжетом, аналізує контекст промпту та планує подальші дії персонажа. Це помітно збільшує загальний обсяг виведеного тексту, але робить фінальне продовження більш логічним і продуманим.

### 0.2. Мої відповіді:
3. Менший словник не призводить до збільшення кількості <unk>, оскільки алгоритм BPE будує токени з базових символів. Якщо зменшити ліміт VOCAB_SIZE, токенізатор зупинить злиття символів раніше і розіб'є довгі слова на дрібніші, вже відомі йому частини. Через це загальна довжина закодованого вірша зростає, і у фіксований CONTEXT_LEN поміститься менше реальних слів

In [ ]:
import numpy as np
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

poems_dataset = load_dataset("RudyiVT/ukrainian-poems", split="train")
poems_df = poems_dataset.to_pandas()

VOCAB_SIZE_SMALL = 4000
SPECIAL_TOKENS = ["<unk>", "<eos>"]

bpe_tokenizer_small = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer_small.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer_small.decoder = decoders.ByteLevel()

bpe_trainer_small = trainers.BpeTrainer(vocab_size=VOCAB_SIZE_SMALL, special_tokens=SPECIAL_TOKENS)

poems_iterator = poems_df["context_poem"].dropna().tolist()
bpe_tokenizer_small.train_from_iterator(poems_iterator, trainer=bpe_trainer_small)

stoi_small = bpe_tokenizer_small.get_vocab()
UNK_ID_SMALL = stoi_small["<unk>"]
EOS_ID_SMALL = stoi_small["<eos>"]

ENCODE_BATCH_SIZE = 1000
token_id_chunks_small = []
total_poems = len(poems_iterator)

for start in range(0, total_poems, ENCODE_BATCH_SIZE):
    batch = poems_iterator[start:start + ENCODE_BATCH_SIZE]
    encodings = bpe_tokenizer_small.encode_batch(batch)
    for enc in encodings:
        token_id_chunks_small.append(np.array(enc.ids + [EOS_ID_SMALL], dtype=np.int32))

token_ids_small = np.concatenate(token_id_chunks_small)

unk_share_small = (token_ids_small == UNK_ID_SMALL).mean()
avg_tokens_per_poem = len(token_ids_small) / total_poems

print(f"Розмір словника: {VOCAB_SIZE_SMALL}")
print(f"Середня кількість токенів на вірш: {avg_tokens_per_poem:.1f}")
print(f"Частка токенів поза словником (<unk>): {unk_share_small:.4%}")

Розмір словника: 4000
Середня кількість токенів на вірш: 1248.1
Частка токенів поза словником (<unk>): 0.0000%


0.2. Мої відповіді:
6.

In [ ]:
assert torch.equal(xb[:, 1:], yb[:, :-1]),


SyntaxError: invalid syntax (2361624787.py, line 1)

---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

In [2]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("Yehor/opentts-uk", "mykyta", split="train")

my_df = dataset.to_pandas()

corpus_text = "\n\n".join(my_df["text"].dropna().astype(str).tolist())

print(f"Завантажено записів: {len(my_df)}")
print(f"Розмір корпусу: {len(corpus_text)} символів")
print("Перші 500 символів корпусу:\n", corpus_text[:500])

README.md:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/3.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Завантажено записів: 6436
Розмір корпусу: 283528 символів
Перші 500 символів корпусу:
 Отож я знаю, що роблю, той ходім.

— А може до якого ґазди?

— Дійсно, це так, — наважився вкинути слово Д'Артаньян.

Троє мушкетерів теж дуже полюбили свого молодого приятеля.

— На нас три, ах, тільки два книдлі?

— Може воно й так.

Потім зв'язав їх до купи майже під прямим кутом.

Ян біг наосліп, бо звучало йому в ушах одне тільки слово жити.

— Як? — скрикнула Марина, — бранці з України?

Тільки в таких випадках слід вживати не шпаду, а ніж.

Пан Іван був значний чоловік, старожитель у М.




---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

In [3]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers
from transformers import AutoTokenizer
import pandas as pd

VOCAB_SIZE = 500
poems_iterator = my_df["text"].dropna().astype(str).tolist()

old_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
qwen_tokenizer = old_tokenizer.train_new_from_iterator(poems_iterator, vocab_size=VOCAB_SIZE)

bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<unk>", "<eos>"])
bpe_tokenizer.train_from_iterator(poems_iterator, trainer=bpe_trainer)

wp_tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
wp_tokenizer.normalizer = normalizers.Sequence([normalizers.Lowercase()])
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
wp_trainer = trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
wp_tokenizer.train_from_iterator(poems_iterator, trainer=wp_trainer)

unigram_tokenizer = Tokenizer(models.Unigram())
unigram_tokenizer.normalizer = normalizers.Sequence([normalizers.Lowercase()])
unigram_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
unigram_trainer = trainers.UnigramTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<unk>", "<pad>", "<cls>", "<sep>", "<mask>"], unk_token="<unk>")
unigram_tokenizer.train_from_iterator(poems_iterator, trainer=unigram_trainer)

sample_text = "Отож я знаю Д'Артаньян"

print(f"Текст: {sample_text}\n")
print(f"Qwen: {qwen_tokenizer.tokenize(sample_text)} (Токенів: {len(qwen_tokenizer.tokenize(sample_text))})")
print(f"BPE: {bpe_tokenizer.encode(sample_text).tokens} (Токенів: {len(bpe_tokenizer.encode(sample_text).tokens)})")
print(f"WordPiece: {wp_tokenizer.encode(sample_text).tokens} (Токенів: {len(wp_tokenizer.encode(sample_text).tokens)})")
print(f"Unigram: {unigram_tokenizer.encode(sample_text).tokens} (Токенів: {len(unigram_tokenizer.encode(sample_text).tokens)})")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Текст: Отож я знаю Д'Артаньян

Qwen: ['Ð', 'ŀ', 'ÑĤ', 'Ð¾Ð¶', 'ĠÑı', 'ĠÐ·Ð½', 'Ð°Ñİ', 'ĠÐĶ', "'ÐĲÑĢÑĤÐ°Ð½ÑĮÑıÐ½"] (Токенів: 9)
BPE: ['Ðŀ', 'ÑĤ', 'Ð¾Ð¶', 'ĠÑı', 'ĠÐ·Ð½', 'Ð°Ñİ', 'ĠÐĶ', "'", 'ÐĲÑĢÑĤÐ°Ð½ÑĮÑıÐ½'] (Токенів: 9)
WordPiece: ['от', '##о', '##ж', 'я', 'зна', '##ю', 'д', "'", 'артаньян'] (Токенів: 9)
Unigram: ['о', 'то', 'ж', 'я', 'знаю', 'д', "'", 'артаньян'] (Токенів: 8)


Qwen та BPE: обидва використовують байтове представлення тексту, через що кирилиця розбивається на нечитабельні багатобайтові символи. Для маленького словника на 500 токенів це робить розбиття кирилиці дуже неефективним візуально, хоча машина його розуміє.

WordPiece: працює безпосередньо з символами кирилиці та використовує префікс ## для субслів. Він розпізнав корінь зна та закінчення ##ю, а слово "Отож" розбив на частини (от, ##о, ##ж). Апостроф та ім'я "артаньян" були виділені коректно.

Unigram: виявився найефективнішим для цієї фрази всього 8 токенів,. Оскільки алгоритм працює від великого словника до малого, він зміг зберегти часте слово знаю як єдиний цілий токен. Слово "Отож" він також розбив на логічні фрагменти (о, то, ж).

---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 32
block_size = 64
max_iters = 1500
eval_interval = 300
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 20
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.1

print(f"Використовується пристрій: {device}")

Використовується пристрій: cpu


In [5]:
vocab_size = unigram_tokenizer.get_vocab_size()

def encode(text):
    return unigram_tokenizer.encode(text).ids

def decode(ids):
    return unigram_tokenizer.decode(ids)

data = torch.tensor(encode(corpus_text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Починаємо тренування...")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Крок {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\n--- Згенерований текст ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_ids = model.generate(context, max_new_tokens=300)[0].tolist()
print(decode(generated_ids))

Починаємо тренування...
Крок 0: train loss 6.3681, val loss 6.3693
Крок 300: train loss 4.8944, val loss 4.8847
Крок 600: train loss 4.5481, val loss 4.6018
Крок 900: train loss 4.3578, val loss 4.4382
Крок 1200: train loss 4.2652, val loss 4.3422
Крок 1499: train loss 4.1654, val loss 4.2881

--- Згенерований текст ---
ні сь по , дійсн те ю . але корол ь па не мо д дозвол а м лі ли у д уже . може , м д но с па но ю . — д ть ми ще в і , ко чи ть ся . — мій те не від па ю мав бу добр еньк ів . льо , певн и ми но хоті . на . всі ш жив с но мій ко т ебе , свої б я н я ви нас не вій земл им до ра с пере ж ь обі май ті в про то в до вел о то й брат оді я ть ла він п ле . з в ко ю ! — ти від ни ми то мо й на ! — на він . з вікн а є й хвилин с ла ще на чудов о , що сь и ти люби ра к ж бо мо ле в мер обі , ви к ня , а ван ну р х . — відповів хто роз у ся , щоб мене й мій ось од но бо да , ш що за бачи ч ку кри в іть га ру ла ся в м а ми вер му ди . і ф ь мі г ре них а так , мі ви й було і д н 

In [6]:
import os
import torch

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/minigpt_v1.pt")
print("Модель v1 успішно збережено!")

Модель v1 успішно збережено!


In [7]:
import os
import torch
from torch.utils.tensorboard import SummaryWriter

n_head = 2
n_layer = 2
max_iters = 1000

print("Ініціалізуємо модель v2 з новими гіперпараметрами")
model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

writer = SummaryWriter('runs/minigpt_v2')

print("Починаємо тренування моделі v2")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Крок {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        writer.add_scalar('Loss/train', losses['train'], iter)
        writer.add_scalar('Loss/val', losses['val'], iter)

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

writer.close()

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/minigpt_v2.pt")
print("\nМодель v2 успішно збережено у файл models/minigpt_v2.pt!")

Ініціалізуємо модель v2 з новими гіперпараметрами
Починаємо тренування моделі v2
Крок 0: train loss 6.3623, val loss 6.3582
Крок 300: train loss 4.9375, val loss 4.9389
Крок 600: train loss 4.6848, val loss 4.7078
Крок 900: train loss 4.4623, val loss 4.5202
Крок 999: train loss 4.4164, val loss 4.4887

Модель v2 успішно збережено у файл models/minigpt_v2.pt!


In [8]:
unigram_tokenizer.save("unigram_tokenizer.json")
print("Токенізатор збережено!")

Токенізатор збережено!


In [9]:
import gc, torch
del model, optimizer, train_data, val_data, data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

### 4.0. Запустіть приклади виконання різних NLP задач

In [1]:
!pip install "transformers[sentencepiece]==5.18.0"

In [2]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}]
[{'label': 'POSITIVE', 'score': 0.9598048329353333}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [3]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445985913276672, 0.11197450757026672, 0.04342687129974365]}

In [4]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to use the correct programming language to create the correct functional programming language.\n\n\n\n\n\nTo use the language, you can now use the following command:\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'}, {'generated_text': 'In this course, we will teach you how to create a single user account, and how to create multiple accounts.\n\n\n\nWe will be looking at our "How to create a single user account" page. In this course, we will examine how you can create multiple accounts, and how to setup a single account. We will examin

In [5]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619713723659515,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040527280420064926,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [6]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [7]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/490 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

{'answer': 'AVERAGE > 36542',
 'coordinates': [(0, 1)],
 'cells': ['36542'],
 'aggregator': 'AVERAGE'}

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

In [8]:
# 1. Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
print(classifier("Книга «Алхімізовані» просто неймовірна, я в повному захваті від магічного світу та розвитку героїв!"))
print(classifier(
    ["Книга «Алхімізовані» просто неймовірна, я в повному захваті від магічного світу та розвитку героїв!",
     "Сюжет здався мені занадто затягнутим, а фінал дуже розчарував."]
))

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': '5 stars', 'score': 0.5948286652565002}]
[{'label': '5 stars', 'score': 0.5948286652565002}, {'label': '3 stars', 'score': 0.458768367767334}]


In [9]:
# 2. Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
classifier(
    "Головна героїня знаходить стародавній грімуар, який відкриває їй таємниці забутої магії та змушує боротися проти темного ордену.",
    candidate_labels=["фентезі", "наука", "детектив", "романтика"],
)

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

{'sequence': 'Головна героїня знаходить стародавній грімуар, який відкриває їй таємниці забутої магії та змушує боротися проти темного ордену.',
 'labels': ['романтика', 'детектив', 'фентезі', 'наука'],
 'scores': [0.6773406863212585,
  0.18185921013355255,
  0.0875130221247673,
  0.0532870776951313]}

In [10]:
# 3. Text generation
from transformers import pipeline

generator = pipeline("text-generation", model="benjamin/gpt2-wechsel-ukrainian")
print(generator("Коли вона вперше розгорнула стародавню книгу алхіміків, з її сторінок"))

print(generator(
    "Коли вона вперше розгорнула стародавню книгу алхіміків, з її сторінок",
    max_length=30,
    num_return_sequences=2,
))

config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  665MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: benjamin/gpt2-wechsel-ukrainian
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.95M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Коли вона вперше розгорнула стародавню книгу алхіміків, з її сторінок з'явилася така незвичайна наука, як астрологія. В основі астрології лежить думка про те, що в світі існує чотири стихії, тобто чотири стихії, які мають однакові сили.\nУ стародавніх книгах астрологія була однією з основних дисциплін, що вивчають астрологічні явища. Знання про астрологія, особливо в стародавньому світі, було дуже важливим для розвитку науки.\nУ Стародавньому Єгипті ще в часи правління фараонів були написані книги про життя богів. У той же час древні греки були переконані, що люди, які живуть в певному суспільстві, дуже добре володіють знаннями про цю особливу сферу. Але в стародавньому Єгипті вони не могли створити точних законів, вони не могли пояснити, що відбувається в природі.\nНа початку XX століття в Росії почалася розробка методів вивчення астрології. Ще в 1913 році, в Ленінграді, була створена група вчених, які проводили експеримент на собаках. Вони вивчили властивості пов

In [11]:
# 4. Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask", model="xlm-roberta-base")
unmasker("Щоб врятувати королівство, головний герой повинен вивчити найсильнішу <mask>.", top_k=2)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

[{'score': 0.7956241965293884,
  'token': 223099,
  'token_str': 'зброю',
  'sequence': 'Щоб врятувати королівство, головний герой повинен вивчити найсильнішу зброю .'},
 {'score': 0.16028693318367004,
  'token': 33969,
  'token_str': 'силу',
  'sequence': 'Щоб врятувати королівство, головний герой повинен вивчити найсильнішу силу .'}]

In [12]:
# 5. Named entity recognition
from transformers import pipeline

ner = pipeline("ner", model="Babelscape/wikineural-multilingual-ner", aggregation_strategy="simple")
ner("Вікторія читала роман 'Алхімізовані', де хоробра героїня подорожує стародавнім містом Ерілея у пошуках артефактів.")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  709MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'entity_group': 'LOC',
  'score': np.float32(0.97706044),
  'word': 'Ерілея',
  'start': 86,
  'end': 92}]

In [13]:
# 6. Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")
question_answerer(
    query="Яку оцінку отримала книга Алхімізовані?",
    table={
        "Книга": ["Алхімізовані", "Жорстокий принц", "Двір тернів і троянд"],
        "Жанр": ["Фентезі", "Дарк фентезі", "Романтичне фентезі"],
        "Оцінка": ["5", "4", "5"]
    },
)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

{'answer': 'Алхімізовані, Двір тернів і троянд',
 'coordinates': [(0, 0), (2, 0)],
 'cells': ['Алхімізовані', 'Двір тернів і троянд'],
 'aggregator': 'NONE'}

---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```